In [1]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import json
import os
import time
import boto3
from pathlib import Path
from datetime import datetime, timezone

In [2]:
print(f"Pandas: {pd.__version__}")
print(f"PyArrow: {pa.__version__}")

Pandas: 3.0.3
PyArrow: 24.0.0


In [4]:
s3_client = boto3.client(
        "s3",
        endpoint_url = "http://localhost:9000",
        aws_access_key_id = "minioadmin",
        aws_secret_access_key = "minioadmin123",
        region_name = "us-east-1"
    )

In [5]:
Path("../data/bronze/iris").mkdir(parents=True, exist_ok=True)
Path("../data/bronze/sintetico").mkdir(parents=True, exist_ok=True)
print("\u2705 Diretórios e conexão MinIO configurados!")

✅ Diretórios e conexão MinIO configurados!


In [6]:
URL_IRIS=("https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data")

In [7]:
COLUNAS_IRIS = [
    "comprimento_sepala_cm",
    "largura_sepala_cm",
    "comprimento_petala_cm",
    "largura_petala_cm",
    "especie",
]

In [9]:
df_iris = pd.read_csv(URL_IRIS, header = None, names = COLUNAS_IRIS)

In [10]:
df_iris["_fonte"] = "uci_iris"

In [11]:
df_iris["_ingerido_em"] = datetime.now(timezone.utc).isoformat()

In [12]:
df_iris["_versao_schema"]="1.0"

In [13]:
print(f"Shape do dataset: {df_iris.shape}")
print(f"\nTipos de dados: \n {df_iris.dtypes}")
print(f"\nPrimeiras 5 linhas:")
df_iris.head()

Shape do dataset: (150, 8)

Tipos de dados: 
 comprimento_sepala_cm    float64
largura_sepala_cm        float64
comprimento_petala_cm    float64
largura_petala_cm        float64
especie                      str
_fonte                       str
_ingerido_em                 str
_versao_schema               str
dtype: object

Primeiras 5 linhas:


,comprimento_sepala_cm,largura_sepala_cm,comprimento_petala_cm,largura_petala_cm,especie,_fonte,_ingerido_em,_versao_schema
0,5.1,3.5,1.4,0.2,Iris-setosa,uci_iris,2026-05-15T00:25:16.237609+00:00,1.0
1,4.9,3.0,1.4,0.2,Iris-setosa,uci_iris,2026-05-15T00:25:16.237609+00:00,1.0
2,4.7,3.2,1.3,0.2,Iris-setosa,uci_iris,2026-05-15T00:25:16.237609+00:00,1.0
3,4.6,3.1,1.5,0.2,Iris-setosa,uci_iris,2026-05-15T00:25:16.237609+00:00,1.0
4,5.0,3.6,1.4,0.2,Iris-setosa,uci_iris,2026-05-15T00:25:16.237609+00:00,1.0


In [14]:
print("+++ Estatísticas Descritivas+++")
print(df_iris.describe())
print(f"\nDistribuição por espécie:")
print(df_iris["especie"].value_counts())

+++ Estatísticas Descritivas+++
       comprimento_sepala_cm  largura_sepala_cm  comprimento_petala_cm  \
count             150.000000         150.000000             150.000000   
mean                5.843333           3.054000               3.758667   
std                 0.828066           0.433594               1.764420   
min                 4.300000           2.000000               1.000000   
25%                 5.100000           2.800000               1.600000   
50%                 5.800000           3.000000               4.350000   
75%                 6.400000           3.300000               5.100000   
max                 7.900000           4.400000               6.900000   

       largura_petala_cm  
count         150.000000  
mean            1.198667  
std             0.763161  
min             0.100000  
25%             0.300000  
50%             1.300000  
75%             1.800000  
max             2.500000  

Distribuição por espécie:
especie
Iris-setosa        50
I

In [15]:
# Criação de um dataset sintético (1 milhão de registros)
print("Gerando dataset sintético de 1.000.000 de registros...")
inicio = time.time()
np.random.seed(42)
N = 1_000_000
CATEGORIAS = ["Eletrônicos", "Roupas", "Alimentos", "Livros", "Esportes"]
STATUS = ["concluido", "cancelado", "pendente", "reembolsado"]
REGIOES = ["Sudeste", "Sul", "Nordeste", "Norte", "Centro-Oeste"]

Gerando dataset sintético de 1.000.000 de registros...


In [16]:
df_sintetico = pd.DataFrame({
    "id_pedido": range(1, N+1),
    "id_cliente": np.random.randint(1, 100_001, N),
    "id_produto": np.random.randint(1, 10_001, N),
    "categoria": np.random.choice(CATEGORIAS, N),
    "valor_unitario": np.round(np.random.uniform(5.0, 2000.0, N),2),
    "quantidade": np.random.randint(1,11,N),
    "status_pedido": np.random.choice(STATUS, N, p=[0.75, 0.10, 0.10, 0.05]),
    "regiao": np.random.choice(REGIOES, N),
    "data_pedido": pd.date_range(start="2022-01-01", periods=N, freq = "30s"),
    "avaliacao_cliente": np.random.choice([1,2,3,4,5,None], N, p=[0.05, 0.08, 0.15, 0.30, 0.37, 0.05]),
    "_fonte": "sistema_ecommerce_v2",
    "_ingerido_em": datetime.now(timezone.utc).isoformat(),
})

In [17]:
df_sintetico["valor_total"] = (df_sintetico["valor_unitario"] * df_sintetico["quantidade"]).round(2)

In [19]:
duracao = time.time() - inicio

In [20]:
print(f"Dataset gerado em {duracao:.2f}s")

Dataset gerado em 88.56s


In [21]:
# Gravação do dataset sintético nos três formatos

BASE_PATH = Path("../data/bronze/sintetico")

# ── CSV ──────────────────────────────────────────────────────────────────────
print("Gravando CSV...")
inicio = time.time()
caminho_csv = BASE_PATH / "pedidos.csv"
df_sintetico.to_csv(caminho_csv, index=False)
tempo_escrita_csv = time.time() - inicio
tamanho_csv = caminho_csv.stat().st_size

# ── JSON (linhas) ─────────────────────────────────────────────────────────────
print("Gravando JSON Lines (JSONL)...")
inicio = time.time()
caminho_json = BASE_PATH / "pedidos.jsonl"
df_sintetico.to_json(caminho_json, orient="records", lines=True, date_format="iso")
tempo_escrita_json = time.time() - inicio
tamanho_json = caminho_json.stat().st_size

# ── Parquet (sem compressão) ──────────────────────────────────────────────────
print("Gravando Parquet (sem compressão)...")
inicio = time.time()
caminho_parquet_raw = BASE_PATH / "pedidos_sem_compressao.parquet"
df_sintetico.to_parquet(caminho_parquet_raw, index=False, compression=None)
tempo_escrita_parquet_raw = time.time() - inicio
tamanho_parquet_raw = caminho_parquet_raw.stat().st_size

# ── Parquet (Snappy — padrão da indústria) ────────────────────────────────────
print("Gravando Parquet (Snappy)...")
inicio = time.time()
caminho_parquet_snappy = BASE_PATH / "pedidos_snappy.parquet"
df_sintetico.to_parquet(caminho_parquet_snappy, index=False, compression="snappy")
tempo_escrita_parquet_snappy = time.time() - inicio
tamanho_parquet_snappy = caminho_parquet_snappy.stat().st_size

# ── Parquet (ZSTD — melhor compressão) ───────────────────────────────────────
print("Gravando Parquet (ZSTD)...")
inicio = time.time()
caminho_parquet_zstd = BASE_PATH / "pedidos_zstd.parquet"
df_sintetico.to_parquet(caminho_parquet_zstd, index=False, compression="zstd")
tempo_escrita_parquet_zstd = time.time() - inicio
tamanho_parquet_zstd = caminho_parquet_zstd.stat().st_size

print("\n✅ Todos os arquivos gravados!")

Gravando CSV...
Gravando JSON Lines (JSONL)...
Gravando Parquet (sem compressão)...
Gravando Parquet (Snappy)...
Gravando Parquet (ZSTD)...

✅ Todos os arquivos gravados!


In [22]:
# Tabela comparativa de tamanho em disco

def formatar_tamanho(bytes_val: int) -> str:
    """Formata bytes em MB com 1 casa decimal."""
    return f"{bytes_val / 1024**2:.1f} MB"

def reducao_percentual(base: int, comparado: int) -> str:
    """Calcula a redução percentual em relação ao CSV."""
    reducao = (1 - comparado / base) * 100
    return f"{reducao:.1f}% menor"

resultados_tamanho = {
    "Formato": ["CSV", "JSON Lines", "Parquet (sem compressão)", "Parquet (Snappy)", "Parquet (ZSTD)"],
    "Tamanho em Disco": [
        formatar_tamanho(tamanho_csv),
        formatar_tamanho(tamanho_json),
        formatar_tamanho(tamanho_parquet_raw),
        formatar_tamanho(tamanho_parquet_snappy),
        formatar_tamanho(tamanho_parquet_zstd),
    ],
    "Redução vs CSV": [
        "— (referência)",
        reducao_percentual(tamanho_csv, tamanho_json),
        reducao_percentual(tamanho_csv, tamanho_parquet_raw),
        reducao_percentual(tamanho_csv, tamanho_parquet_snappy),
        reducao_percentual(tamanho_csv, tamanho_parquet_zstd),
    ],
    "Tempo de Escrita (s)": [
        f"{tempo_escrita_csv:.2f}",
        f"{tempo_escrita_json:.2f}",
        f"{tempo_escrita_parquet_raw:.2f}",
        f"{tempo_escrita_parquet_snappy:.2f}",
        f"{tempo_escrita_parquet_zstd:.2f}",
    ],
}

df_tamanhos = pd.DataFrame(resultados_tamanho)
print("=== Comparativo de Tamanho em Disco (1.000.000 registros) ===")
print(df_tamanhos.to_string(index=False))

=== Comparativo de Tamanho em Disco (1.000.000 registros) ===
                 Formato Tamanho em Disco Redução vs CSV Tempo de Escrita (s)
                     CSV         132.6 MB — (referência)                21.34
              JSON Lines         318.5 MB  -140.3% menor                12.87
Parquet (sem compressão)          37.1 MB    72.1% menor                 1.60
        Parquet (Snappy)          27.1 MB    79.6% menor                 1.42
          Parquet (ZSTD)          19.8 MB    85.1% menor                 1.50


In [23]:
REPETICOES = 3
def medir_tempo_leitura(funcao_leitura, repeticoes=REPETICOES):
    """Executa a função de leitura N vezes e retorna o tempo médio.""" 
    tempos = [] 
    for _ in range(repeticoes): 
        inicio = time.time() 
        funcao_leitura() 
        tempos.append(time.time() - inicio) 
        return sum(tempos) / len(tempos)

In [24]:
tempo_csv_completo = medir_tempo_leitura(
    lambda: pd.read_csv(caminho_csv) 
)

In [25]:
tempo_json_completo = medir_tempo_leitura( 
    lambda: pd.read_json(caminho_json, lines=True) 
) 

In [26]:
tempo_parquet_completo = medir_tempo_leitura( 
    lambda: pd.read_parquet(caminho_parquet_snappy) 
)

In [27]:
print(f"Leitura COMPLETA (1.000.000 linhas, todas as colunas):") 
print(f" CSV: {tempo_csv_completo:.3f}s") 
print(f" JSON: {tempo_json_completo:.3f}s") 
print(f" Parquet: {tempo_parquet_completo:.3f}s") 
print(f" → Parquet é {tempo_csv_completo / tempo_parquet_completo:.1f}x mais rápido que CSV")

Leitura COMPLETA (1.000.000 linhas, todas as colunas):
 CSV: 5.367s
 JSON: 18.855s
 Parquet: 1.624s
 → Parquet é 3.3x mais rápido que CSV


In [28]:
# Célula 9 — Leitura seletiva de colunas (column pruning) 
# Esta é a vantagem FUNDAMENTAL do formato colunar 
COLUNAS_ANALITICAS = ["categoria", "valor_total", "status_pedido", "regiao"] 
# CSV: precisa ler TODAS as colunas e depois filtrar 
tempo_csv_seletivo = medir_tempo_leitura( 
    lambda: pd.read_csv(caminho_csv, usecols=COLUNAS_ANALITICAS) 
) 
# Parquet: lê APENAS as colunas solicitadas do disco 
tempo_parquet_seletivo = medir_tempo_leitura( 
    lambda: pd.read_parquet(caminho_parquet_snappy, 
    columns=COLUNAS_ANALITICAS) 
) 
print(f"Leitura SELETIVA (apenas 4 de 14 colunas):") 
print(f" CSV: {tempo_csv_seletivo:.3f}s (ainda lê tudo do disco)") 
print(f" Parquet: {tempo_parquet_seletivo:.3f}s (lê apenas as colunas necessárias)") 
print(f" → Parquet é {tempo_csv_seletivo / tempo_parquet_seletivo:.1f}x mais rápido na leitura seletiva") 
print() 
print("Em Data Lakes com petabytes de dados, essa diferença representa") 
print(" economia de horas de processamento e centenas de dólares em custo de nuvem.")

Leitura SELETIVA (apenas 4 de 14 colunas):
 CSV: 2.973s (ainda lê tudo do disco)
 Parquet: 0.111s (lê apenas as colunas necessárias)
 → Parquet é 26.7x mais rápido na leitura seletiva

Em Data Lakes com petabytes de dados, essa diferença representa
 economia de horas de processamento e centenas de dólares em custo de nuvem.


In [29]:
# Célula 10 — Inspeção do schema do arquivo Parquet 
schema_parquet = pq.read_schema(caminho_parquet_snappy) 
print("=== Schema do Arquivo Parquet ===") 
print(schema_parquet) 
print() 
print("=== Metadados do Arquivo ===") 
metadata = pq.read_metadata(caminho_parquet_snappy) 
print(f"Número de row groups: {metadata.num_row_groups}") 
print(f"Número de colunas: {metadata.num_columns}") 
print(f"Número de linhas: {metadata.num_rows:,}") 
print(f"Tamanho serializado: {metadata.serialized_size:,} bytes") 
print() 
print("O Parquet armazena o schema junto com os dados.") 
print(" Isso elimina a necessidade de inferência de tipos na leitura,") 
print(" garantindo consistência e evitando erros silenciosos.")

=== Schema do Arquivo Parquet ===
id_pedido: int64
id_cliente: int32
id_produto: int32
categoria: large_string
valor_unitario: double
quantidade: int32
status_pedido: large_string
regiao: large_string
data_pedido: timestamp[us]
avaliacao_cliente: int64
_fonte: large_string
_ingerido_em: large_string
valor_total: double
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1646

=== Metadados do Arquivo ===
Número de row groups: 1
Número de colunas: 13
Número de linhas: 1,000,000
Tamanho serializado: 7,090 bytes

O Parquet armazena o schema junto com os dados.
 Isso elimina a necessidade de inferência de tipos na leitura,
 garantindo consistência e evitando erros silenciosos.


In [30]:
# Célula 11 - Upload do parquet para o MinIO (camada Bronze)
hoje = datetime.now()
prefixo_particao = f"ano={hoje.year}/mes={hoje.month:02d}/dia= {hoje.day:02d}" 
# Upload do Parquet Snappy para o bucket Bronze 
chave_objeto = f"ecommerce_sintetico/{prefixo_particao}/pedidos.parquet" 
print(f"Fazendo upload para: bronze/{chave_objeto}") 
inicio = time.time() 
s3_client.upload_file( 
    Filename=str(caminho_parquet_snappy),
    Bucket="bronze", 
    Key=chave_objeto, 
    ExtraArgs={"ContentType": "application/octet-stream"}, 
)
duracao = time.time() - inicio 
print(f"✅ Upload concluído em {duracao:.2f}s") 
# Verificar o objeto no MinIO 
response = s3_client.head_object(Bucket="bronze", Key=chave_objeto) 
print(f"\nMetadados no MinIO:") 
print(f" Tamanho: {response['ContentLength'] / 1024**2:.1f} MB") 
print(f" Última modificação: {response['LastModified']}") 
print(f"\nDado bruto armazenado na camada Bronze do Data Lake!") 
print(f" Acesse em: http://localhost:9001/browser/bronze")

Fazendo upload para: bronze/ecommerce_sintetico/ano=2026/mes=05/dia= 14/pedidos.parquet
✅ Upload concluído em 6.10s

Metadados no MinIO:
 Tamanho: 27.1 MB
 Última modificação: 2026-05-15 00:55:22+00:00

Dado bruto armazenado na camada Bronze do Data Lake!
 Acesse em: http://localhost:9001/browser/bronze


In [ ]:
# Célula 12 — Resumo final comparativo 
print("=" * 65) 
print(" RESUMO COMPARATIVO — FORMATOS DE ARQUIVO PARA DATA LAKES") 
print("=" * 65) 
resumo = pd.DataFrame({
    "Característica": [ 
        "Tipo de armazenamento", 
        "Compressão nativa", 
        "Schema embutido", 
        "Leitura seletiva de colunas", 
        "Suporte a tipos complexos", 
        "Legível por humanos", 
        "Ideal para Data Lakes", 
        "Suporte em ferramentas", 
    ], 
    "CSV": [ 
        "Orientado a linhas", 
        "Não", 
        "Não (inferido)", 
        "Não (lê tudo)", 
        "Não", 
        "Sim", 
        "❌ Não recomendado", 
        "Universal", 
    ], 
        "JSON/JSONL": [ 
    "Orientado a linhas", 
    "Não", 
    "Não (inferido)", 
    "Não (lê tudo)", 
    "Sim (aninhado)", 
    "Sim", 
    "⚠️ Apenas para APIs", 
    "Universal", 
    ], 
    "Parquet": [ 
        "Orientado a colunas", 
        "Sim (Snappy/ZSTD)", 
        "Sim (forte tipagem)", 
        "Sim (column pruning)", 
        "Sim (listas, mapas)",
        "Não (binário)", 
        "✅ Padrão da indústria", 
        "Spark, DuckDB, Iceberg...", 
    ], 
    }) 
print(resumo.to_string(index=False)) 
print() 
print("Conclusão: O formato Parquet com compressão Snappy ou ZSTD é o") 
print("padrão da indústria para Data Lakes por combinar alta compressão,")
print("leitura seletiva de colunas e schema fortemente tipado.")